# Preprocessing and experiment design

Project 18: reproduce the actual partitions and training-only transforms. Run EDA first to understand the target and source limitations.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image
ROOT = Path.cwd() if (Path.cwd() / 'configs/default.json').exists() else Path.cwd().parent
REPORTS = ROOT / 'reports'
config = json.loads((ROOT / 'configs/default.json').read_text())


## 1. Clean metadata and create chronological partitions

Never impute the target or use random row splitting for this benchmark.

In [2]:
from water_quality.data import read_dataset, prepare, temporal_split, sample, quality_features, FEATURES
raw = read_dataset(ROOT / config['data'])
clean, removed = prepare(raw)
parts = temporal_split(clean, config)
print('Removed:', removed)
display(pd.DataFrame({name: {'rows':len(part), 'start':str(part.parsed_date.min().date()), 'end':str(part.parsed_date.max().date())} for name,part in parts.items()}).T)

Removed: {'exact_duplicates_removed': 9971, 'invalid_metadata_or_target': 0}


,rows,start,end
train,2478761,1940-01-02,2018-12-28
validation,87177,2019-01-01,2019-12-31
calibration,49182,2020-01-01,2020-12-31
test,202886,2021-01-01,2023-12-17


## 2. Share the exact training sample across models

The same seeded 300,000 rows train all model families. Candidate tuning uses a seeded 50,000-row validation subset. The complete validation period chooses the family and blend.

In [3]:
train = sample(parts['train'], config['max_train_rows'], config['seed'])
tune = sample(parts['validation'], config['max_tuning_rows'], config['seed'])
x = quality_features(train)
print('Training rows:', len(train), 'Tuning rows:', len(tune))
print('Model input dimensions:', x.shape[1])
display(x.head())

Training rows: 300000 Tuning rows: 50000
Model input dimensions: 16


,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l),Ammonia (mg/l)__missing,Biochemical Oxygen Demand (mg/l)__missing,Dissolved Oxygen (mg/l)__missing,Orthophosphate (mg/l)__missing,pH (ph units)__missing,Temperature (cel)__missing,Nitrogen (mg/l)__missing,Nitrate (mg/l)__missing
0,0.059248,1.3000,8.150000,0.011917,8.07500,9.885,0.343917,11.73155,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.039821,1.3800,7.800000,0.009417,7.73333,10.150,0.449083,11.82009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11,0.020823,2.3200,0.183333,0.006833,7.13333,10.650,0.418333,19.12464,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12,0.038855,2.6200,7.650000,0.008750,7.68000,8.069,0.336167,18.01789,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
26,0.051520,3.9076,11.684103,0.010000,8.37000,10.000,0.241667,9.73940,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Verify training-fitted preprocessing

Invalid inputs become missing and receive fixed flags. Tree models use median-imputed values. MLP/Ridge add asinh and standard scaling. Loading joblib is safe only for trusted local artifacts.

In [4]:
import joblib
pre = joblib.load(ROOT / 'artifacts/preprocessing.joblib')
manual_medians = np.nanmedian(x.to_numpy(), axis=0)
manual_medians = np.nan_to_num(manual_medians, nan=0)
np.testing.assert_allclose(pre['tree'].statistics_, manual_medians)
xt = pre['tree'].transform(x.iloc[:5])
xn = pre['neural'].transform(x.iloc[:5])
print('Tree transform shape:', xt.shape, 'Neural shape:', xn.shape)
print('Stored medians match the historical training sample.')

Tree transform shape: (5, 16) Neural shape: (5, 16)
Stored medians match the historical training sample.


## 4. Verify disjoint rows and chronological boundaries

Historical stations can recur. Strictly separated dates prevent future rows from entering historical fitting, but cannot undo preprocessing already performed by the dataset authors.

In [5]:
for name,part in parts.items():
    print(name, len(part), part.parsed_date.min(), part.parsed_date.max())
sets = [set(part.source_row) for part in parts.values()]
assert sum(map(len,sets)) == len(set.union(*sets))
ordered = list(parts.values())
assert all(left.parsed_date.max() < right.parsed_date.min() for left,right in zip(ordered, ordered[1:]))
print('All partitions have disjoint source rows and ordered dates.')

train 2478761 1940-01-02 00:00:00 2018-12-28 00:00:00
validation 87177 2019-01-01 00:00:00 2019-12-31 00:00:00
calibration 49182 2020-01-01 00:00:00 2020-12-31 00:00:00
test 202886 2021-01-01 00:00:00 2023-12-17 00:00:00
All partitions have disjoint source rows and ordered dates.
